# System Dependencies

To get started with Unstructured.io, we need a few system-wide dependencies:

## Poppler (poppler-utils)

Handles PDF processing. It's a library that can extract text, images, and metadata from PDFs. Unstructured uses it to parse PDF documents and convert them into processable text.

## Tesseract (tesseract-ocr)

Optical Character Recognition (OCR) engine. When you have scanned documents, images with text, or PDFs that are essentially pictures, Tesseract reads the text from these images and converts it to machine-readable text.

## libmagic

File type detection library. It identifies what type of file you're dealing with (PDF, Word doc, image, etc.) by analyzing the file's content, not just the extension. This helps Unstructured choose the right processing method for each document.

In [1]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain-chroma langchain langchain-community langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

import json
from typing import List
from dotenv import load_dotenv

# Unstructured for document parsing
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

def partition_document(file_path: str):
    print(f"📄 Partitioning document: {file_path}")
    elements = partition_pdf(
        filename=file_path,
        strategy="hi_res",
        infer_table_structure=True,
        extract_image_block_types=["Image"],
        extract_image_block_to_payload=True
    )
    print(f"✅ Extracted {len(elements)} elements")
    return elements

file_path = "./docs/attention-is-all-you-need.pdf"
elements = partition_document(file_path)

/Users/spender/Coding Folder/Rag/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/spender/Coding Folder/Rag/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📄 Partitioning document: ./docs/attention-is-all-you-need.pdf


The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


✅ Extracted 215 elements


In [3]:
tables = [e for e in elements if e.category == "Table"]
tables[0].to_dict()

{'type': 'Table',
 'element_id': 'a89281d85d3d589d6b9b2c889815cbce',
 'text': 'Layer Type Complexity per Layer Sequential Maximum Path Length Operations Self-Attention O(n2 · d) O(1) O(1) Recurrent O(n · d2) O(n) O(n) Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r)',
 'metadata': {'detection_class_prob': 0.9281627535820007,
  'coordinates': {'points': ((np.float64(320.29791259765625),
     np.float64(314.0191955566406)),
    (np.float64(320.29791259765625), np.float64(519.3814697265625)),
    (np.float64(1367.6685791015625), np.float64(519.3814697265625)),
    (np.float64(1367.6685791015625), np.float64(314.0191955566406))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-10-01T22:46:43',
  'text_as_html': '<table><thead><tr><th>Layer Type</th><th>Complexity per Layer</th><th>Sequential Operations</th><th>Maximum Path Length</th></tr></thead><tbody><tr><td>Self-Attention</td><td>O(n

In [4]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    chunks = chunk_by_title(
        elements,
        max_characters=3000,       # Maksimum 3000 karakter
        new_after_n_chars=2400,     # 2400'den sonra yeni başlık ara
        combine_text_under_n_chars=500  # 500 karakterden küçükleri birleştir
    )
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 25 chunks


In [5]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data


def create_ai_enhanced_summary(text: str, tables: List[str], images: List[str]) -> str:
    """Create AI-enhanced summary for mixed content using GPT-4o"""
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        
        prompt_text = f"""You are creating a searchable description for document content retrieval.

CONTENT TO ANALYZE:
TEXT CONTENT:
{text}
"""
        if tables:
            prompt_text += "\nTABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"
        
        prompt_text += """
YOUR TASK:
Generate a comprehensive, searchable description that covers:
1. Key facts, numbers, and data points from text and tables
2. Main topics and concepts discussed  
3. Questions this content could answer
4. Visual content analysis (charts, diagrams in images)
5. Alternative search terms users might use

Make it detailed and searchable.
SEARCHABLE DESCRIPTION:"""

        message_content = [{"type": "text", "text": prompt_text}]
        
        # Resimleri ekle (Base64)
        for image_base64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })
        
        response = llm.invoke([HumanMessage(content=message_content)])
        return response.content
        
    except Exception as e:
        print(f"     ❌ AI summary failed: {e}")
        return text


def summarise_chunks(chunks):
    """Process all chunks and create LangChain Documents"""
    print("🧠 Processing chunks with AI Summaries...")
    langchain_documents = []
    
    for i, chunk in enumerate(chunks, 1):
        print(f"   Processing chunk {i}/{len(chunks)}")
        content_data = separate_content_types(chunk)
        
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            enhanced_content = create_ai_enhanced_summary(
                content_data['text'], content_data['tables'], content_data['images']
            )
        else:
            enhanced_content = content_data['text']
        
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        langchain_documents.append(doc)
    
    print(f"✅ Processed {len(langchain_documents)} chunks")
    return langchain_documents

In [7]:
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 1. Chunk'ları AI ile özetle (HumanMessage artık tanımlı olduğu için özetler üretilecek!)
processed_chunks = summarise_chunks(chunks)

# 2. ChromaDB'ye kaydet
def create_vector_store(documents, persist_directory="dbv2/chroma_db"):
    print("\n🔮 Creating embeddings and storing in ChromaDB...")
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )
    print(f"✅ Vector store saved to {persist_directory}")
    return vectorstore

db = create_vector_store(processed_chunks)

🧠 Processing chunks with AI Summaries...
   Processing chunk 1/25
   Processing chunk 2/25
   Processing chunk 3/25
   Processing chunk 4/25
   Processing chunk 5/25
     → Creating AI summary for mixed content...
   Processing chunk 6/25
   Processing chunk 7/25
     → Creating AI summary for mixed content...
   Processing chunk 8/25
   Processing chunk 9/25
   Processing chunk 10/25
   Processing chunk 11/25
   Processing chunk 12/25
     → Creating AI summary for mixed content...
   Processing chunk 13/25
   Processing chunk 14/25
   Processing chunk 15/25
   Processing chunk 16/25
   Processing chunk 17/25
     → Creating AI summary for mixed content...
   Processing chunk 18/25
   Processing chunk 19/25
     → Creating AI summary for mixed content...
   Processing chunk 20/25
     → Creating AI summary for mixed content...
   Processing chunk 21/25
   Processing chunk 22/25
   Processing chunk 23/25
   Processing chunk 24/25
   Processing chunk 25/25
     → Creating AI summary for

In [9]:
def generate_final_answer(chunks, query):
    """Generate final answer using text + tables + original images"""
    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0)
        prompt_text = f"Based on the following documents, please answer this question: {query}\n\nCONTENT TO ANALYZE:\n"
        
        for i, chunk in enumerate(chunks, 1):
            prompt_text += f"--- Document {i} ---\n"
            if "original_content" in chunk.metadata:
                original_data = json.loads(chunk.metadata["original_content"])
                raw_text = original_data.get("raw_text", "")
                if raw_text:
                    prompt_text += f"TEXT:\n{raw_text}\n\n"
                tables_html = original_data.get("tables_html", [])
                for j, table in enumerate(tables_html, 1):
                    prompt_text += f"Table {j}:\n{table}\n\n"
        
        prompt_text += "\nPlease provide a clear answer using the text, tables, and images. ANSWER:"
        message_content = [{"type": "text", "text": prompt_text}]
        
        # Orijinal görselleri mesaja ekle
        for chunk in chunks:
            if "original_content" in chunk.metadata:
                images_base64 = json.loads(chunk.metadata["original_content"]).get("images_base64", [])
                for img in images_base64:
                    message_content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img}"}
                    })
        
        response = llm.invoke([HumanMessage(content=message_content)])
        return response.content
    except Exception as e:
        return f"Error: {e}"

# Test edelim!
query = "Bu dokuman ne dokumani neyi anlatiyor? bana bazi resimler gosterir misin ve tablolarin icindeki verileri de aciklar misin?"
retriever = db.as_retriever(search_kwargs={"k": 3})
retrieved_chunks = retriever.invoke(query)

answer = generate_final_answer(retrieved_chunks, query)
print("\n🎯 CEVAP:\n", answer)


🎯 CEVAP:
 Bu belgeler, Transformer modelinin farklı varyasyonlarını ve bu varyasyonların İngilizce-Almanca çeviri üzerindeki performansını inceleyen bir çalışmayı anlatıyor. Transformer modelinin bileşenlerinin önemini değerlendirmek için çeşitli model değişiklikleri yapılmış ve sonuçlar karşılaştırılmış.

**Tablo 3'teki Veriler:**
- Model varyasyonları, dikkat başlıklarının sayısı, dikkat anahtar ve değer boyutları gibi parametrelerde değişiklikler yapılarak incelenmiş.
- Performans ölçütleri arasında kelime başına düşen karmaşıklık (PPL) ve BLEU skoru yer alıyor.
- Daha fazla başlık sayısı her zaman daha iyi sonuç vermemiş, aşırı başlık sayısı kaliteyi düşürmüş.
- Daha büyük modeller genellikle daha iyi performans göstermiş ve dropout kullanımı aşırı öğrenmeyi önlemede faydalı olmuş.

**Görseller:**
- Görseller, dikkat mekanizmasının nasıl çalıştığını ve cümle yapısındaki uzun mesafeli bağımlılıkları nasıl takip ettiğini gösteriyor.
- Farklı renkler, farklı dikkat başlıklarını temsi